In [ ]:
import numpy as np
import scipy as sp
import matplotlib.pyplot as plt
from scipy.stats import linregress as LR
import pandas as pd
from pandas.util.testing import assert_frame_equal
from sklearn.model_selection import train_test_split
pd.set_option("display.max_columns",200)

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from subprocess import call

import xgboost as xgb

from sklearn.utils import resample
from sklearn.metrics import recall_score

In [ ]:
#make a chronic patient table for this subset of conditions

# read in that pre-made table
df=pd.read_csv("../data_files/chronic_patients.csv")
df = df.dropna(axis = 0,how='any')
upsample=False

In [ ]:
X = df[['VMONTH', 'AGE', 'SEX', 'RACE', 'ETHNIC', 'RFV1', 'DIAG1', 'DIAG2',
              'DIAG3', 'DIAG4', 'DIAG5', 'PAYTYPE','YEAR','PATWT', 'chronic','pain']]
y = df[['received']]

X_train, X_test, y_train, y_test = train_test_split(X, y)

train = pd.concat([X_train, y_train], axis=1, sort=False)

In [ ]:
#upsample the opiate-receivers
df_maj = train[train.received==0]
df_min = train[train.received==1]

if upsample:
    df_minority_upsampled = resample(df_min, 
                                     replace=True,     # sample with replacement
                                     n_samples=len(df_maj))    # to match majority class
                                     #random_state=123) # reproducible results
    # Combine majority class with upsampled minority class
    train = pd.concat([df_maj, df_minority_upsampled])
else: #downsample
    df_majority_downsampled = resample(df_maj, 
                                     replace=False,     # sample with replacement
                                     n_samples=len(df_min))    # to match majority class
                                     #random_state=123) # reproducible results
    # Combine minority class with downsampled majority class
    train = pd.concat([df_majority_downsampled, df_min])

 
# Display new class counts
train.received.value_counts()

In [ ]:
# upsample the pain-diagnosed
df_maj = train[train.pain==0]
df_min = train[train.pain==1]
df_minority_upsampled = resample(df_min, 
                                 replace=True,     # sample with replacement
                                 n_samples=len(df_maj),    # to match majority class
                                 random_state=123) # reproducible results
# Combine majority class with upsampled minority class
train = pd.concat([df_maj, df_minority_upsampled])
 
# Display new class counts
train.pain.value_counts()

In [ ]:
X_train = train[['VMONTH', 'AGE', 'SEX', 'RACE', 'ETHNIC', 'RFV1', 'DIAG1', 'DIAG2',
              'DIAG3', 'DIAG4', 'DIAG5', 'PAYTYPE','YEAR','PATWT', 'chronic','pain']]
y_train = train[['received']]

In [ ]:
model = xgb.XGBClassifier()#learning_rate=0.01)

In [ ]:
model.fit(X_train[['VMONTH', 'AGE', 'SEX', 'RACE', 'ETHNIC', 'RFV1', 'DIAG1', 'DIAG2',
              'DIAG3', 'DIAG4', 'DIAG5', 'PAYTYPE','YEAR', 'chronic','pain']], y_train[['received']],sample_weight=X_train.PATWT.ravel())

In [ ]:
y_predict = model.predict(X_test[['VMONTH', 'AGE', 'SEX', 'RACE', 'ETHNIC', 'RFV1', 'DIAG1', 'DIAG2',
              'DIAG3', 'DIAG4', 'DIAG5', 'PAYTYPE','YEAR', 'chronic','pain']])#,sample_weight=X_wt2.PATWT.ravel())

In [ ]:
accuracy_score(y_test, y_predict,sample_weight=X_test.PATWT.ravel())

In [ ]:
pd.DataFrame(
    confusion_matrix(y_test[['received']], y_predict),
    columns=['Predicted Not Received', 'Predicted Received'],
    index=['True Not Received', 'True Received']
)

In [ ]:
recall_score(y_test, y_predict)

In [ ]:
plt.bar(range(len(model.feature_importances_)), model.feature_importances_)
plt.xticks(list(range(15)),['VMONTH', 'AGE', 'SEX', 'RACE', 'ETHNIC', 'RFV1', 'DIAG1', 'DIAG2',
              'DIAG3', 'DIAG4', 'DIAG5', 'PAYTYPE','YEAR', 'chronic','pain'], rotation=30)
plt.show()